# Kazakh Profanity Detection — 03. Inference

Loads the trained model and predicts whether a Kazakh comment contains profanity. **No training and no dataset are needed.**

- Output: `label` (`0` = no Kazakh profanity, `1` = profanity present) and `probability` of the profanity class.
- Set `MODEL_ID` to the Hugging Face Hub repository of the model, or to a local folder containing the saved model, tokenizer and `inference_config.json`.
- Long comments are truncated to 128 tokens. Phone numbers, links, e-mails and @-handles are masked exactly as during training.
- A GPU is not required.

In [ ]:
!pip install -q "transformers==5.22.1"

In [ ]:
import json
import os
import re

import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Hub repo id, or a local folder with the saved model,
# e.g. "/content/drive/MyDrive/kazakh_profanity_model" to test the model saved on Google Drive
MODEL_ID = "karlygaw/kazakh-profanity-xlmr"

if MODEL_ID.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"


def mask_pii(text):
    """Same personal-data masking as in the training data."""
    text = re.sub(r"https?://\S+|www\.\S+", "<URL>", text)
    text = re.sub(r"\S+@\S+\.\S+", "<EMAIL>", text)
    text = re.sub(r"@\w+", "<USER>", text)
    return re.sub(r"\+?\d[\d\-\s\(\)]{8,}\d", "<PHONE>", text)


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device).eval()
    if os.path.isdir(model_id):
        cfg_path = os.path.join(model_id, "inference_config.json")
    else:
        cfg_path = hf_hub_download(model_id, "inference_config.json")
    with open(cfg_path) as fh:
        config = json.load(fh)
    return tokenizer, model, config


def predict(text, tokenizer, model, config):
    """Classify one comment. Returns the label and the probability of profanity."""
    enc = tokenizer(mask_pii(str(text)).strip(), truncation=True,
                    max_length=config["max_len"], return_tensors="pt").to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    return {"label": int(prob >= config["threshold"]), "probability": round(prob, 3)}


tokenizer, model, config = load_model(MODEL_ID)
print("Loaded. Decision threshold:", config["threshold"])

config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

inference_config.json:   0%|          | 0.00/75.0 [00:00<?, ?B/s]

Loaded. Decision threshold: 0.7


## Examples

In [ ]:
examples = [
    "Өте жақсы қосымша, маған ұнады",           # positive review
    "Қосымша дұрыс жұмыс істемейді, өте нашар",  # negative review, no profanity
    "piiiiii$$$$daaa, қайтадан қате шықты",                # Russian obscenity inside Kazakh text
]
for text in examples:
    print(predict(text, tokenizer, model, config), "|", text)

{'label': 0, 'probability': 0.003} | Өте жақсы қосымша, маған ұнады
{'label': 0, 'probability': 0.004} | Қосымша дұрыс жұмыс істемейді, өте нашар
{'label': 1, 'probability': 0.798} | piiiiii$$$$daaa, қайтадан қате шықты


## Try your own text

Run the cell and type a comment. An empty line stops the loop.

In [ ]:
while True:
    text = input("Comment (empty line to stop): ")
    if not text.strip():
        break
    print(predict(text, tokenizer, model, config))